In [1]:
# ======================================================
# IMPORT LIBRARY
# ======================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from imblearn.combine import SMOTETomek

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# ======================================================
# IMPORT LIBRARY
# ======================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from imblearn.combine import SMOTETomek

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# ======================================================
# LOAD DATA
# ======================================================

data_path = Path("data") / "matches.xlsx"
df = pd.read_excel(data_path)

print("Shape data awal:", df.shape)

Shape data awal: (19847, 88)


In [4]:
# ======================================================
# FILTER FOOTBALL ONLY
# ======================================================

df = df[df["match_main_genre"].astype(str).str.lower() == "football"]

print("Shape setelah filter Football:", df.shape)

Shape setelah filter Football: (6107, 88)


In [5]:
# ======================================================
# PILIH KOLOM
# ======================================================

cols = [
    "match_date_start",
    "match_duration",
    "match_tournament",
    "match_premier_status",
    "match_age_rating",
    "match_content_type",
    "match_coverage",
    "match_genre",
    "match_main_genre",
    "match_channel",
    "match_gender",
    "match_organization",
    "team_home",
    "team_away",
    "match_priority_level"
]

df = df[cols].copy()

In [7]:
# ======================================================
# FEATURE ENGINEERING
# ======================================================

df["match_date_start"] = pd.to_datetime(df["match_date_start"])

df["match_hour"] = df["match_date_start"].dt.hour
df["match_day"] = df["match_date_start"].dt.dayofweek
df["match_month"] = df["match_date_start"].dt.month

df["is_weekend"] = df["match_day"].isin([5,6]).astype(int)
df["is_prime_time"] = df["match_hour"].between(18,23).astype(int)

df.drop(columns=["match_date_start"], inplace=True)

In [9]:
# ======================================================
# CLEAN TARGET
# ======================================================

df["match_priority_level"] = (
    df["match_priority_level"]
    .astype(str)
    .str.strip()
    .str.lower()
)

priority_map = {
    "low":0,
    "medium":1,
    "high":2
}

df["match_priority_level"] = df["match_priority_level"].map(priority_map)

df = df.dropna(subset=["match_priority_level"])

print("\nDistribusi target:")
print(df["match_priority_level"].value_counts())


Distribusi target:
match_priority_level
1.0    5335
2.0     468
0.0     289
Name: count, dtype: int64


In [10]:
# ======================================================
# ENCODING CATEGORICAL
# ======================================================

cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))


# ======================================================
# SPLIT FEATURE & TARGET
# ======================================================

X = df.drop(columns=["match_priority_level"])
y = df["match_priority_level"]

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17516\1462716524.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = le.fit_transform(df[col].astype(str))
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17516\1462716524.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = le.fit_transform(df[col].astype(str))
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17516\1462716524.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,c

In [12]:
# ======================================================
# TRAIN TEST SPLIT
# ======================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)



Train shape: (4873, 18)
Test shape: (1219, 18)


In [13]:
# ======================================================
# BALANCING DATA (SMOTETomek)
# ======================================================

smote_tomek = SMOTETomek(
    sampling_strategy="not majority",
    random_state=42
)

X_train_res, y_train_res = smote_tomek.fit_resample(
    X_train,
    y_train
)

print("\nDistribusi setelah SMOTETomek:")
print(pd.Series(y_train_res).value_counts())


Distribusi setelah SMOTETomek:
match_priority_level
0.0    4251
2.0    4242
1.0    4227
Name: count, dtype: int64


In [14]:
# ======================================================
# RANDOM FOREST MODEL
# ======================================================

rf_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_res, y_train_res)

rf_pred = rf_model.predict(X_test)

print("\n===== RANDOM FOREST =====")

print("Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))




===== RANDOM FOREST =====
Accuracy: 0.9015586546349467
              precision    recall  f1-score   support

         0.0       0.64      0.90      0.75        58
         1.0       0.97      0.92      0.94      1067
         2.0       0.53      0.73      0.61        94

    accuracy                           0.90      1219
   macro avg       0.71      0.85      0.77      1219
weighted avg       0.92      0.90      0.91      1219



In [15]:
# ======================================================
# XGBOOST MODEL
# ======================================================

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

xgb_model.fit(X_train_res, y_train_res)

xgb_pred = xgb_model.predict(X_test)

print("\n===== XGBOOST =====")

print("Accuracy:", accuracy_score(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))


===== XGBOOST =====
Accuracy: 0.9237079573420837
              precision    recall  f1-score   support

         0.0       0.68      0.86      0.76        58
         1.0       0.96      0.95      0.96      1067
         2.0       0.67      0.66      0.66        94

    accuracy                           0.92      1219
   macro avg       0.77      0.82      0.79      1219
weighted avg       0.93      0.92      0.92      1219

